# 02 Models and parameter counts

**Purpose.** Build the hybrid model (8-qubit circuit, 4 layers, data re-uploading), the parameter-matched classical bottlenecks and the classical twin from one factory, and show that the classical encoder and decoder are identical in every variant.

**Inputs.** `src/models.py`, `src/circuits.py`. **Outputs.** Parameter and footprint tables (paper: parameter-budget and ablation tables).

In [1]:
# --- setup: work from the laptop/ folder so that relative paths resolve ---
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
pd.set_option("display.precision", 4)
print("working folder:", ROOT)

working folder: <home>\Documents\Marwen_quintuimMachine\Quantium machine submission\journal_of_network_and_computer_applications\revision round 2\github_repo\QMedShield\laptop


In [2]:
from models import build_model
rows = []
for task in ["binary", "multiclass"]:
    for variant, label in [("full", "Hybrid (quantum circuit)"), ("linear", "Classical twin (dense 8x8 + tanh)"),
                           ("mlp96", "Tiny MLP (93 params)"), ("fourier", "Random Fourier map"), ("identity", "Identity (no bottleneck)")]:
        m = build_model(task, variant)
        b = m.param_breakdown()
        rows.append({"task": task, "model": label, **b, "footprint KB (32 bit)": round(m.footprint_kb(), 1)})
pd.DataFrame(rows)

,task,model,encoder,bottleneck,decoder,total,footprint KB (32 bit)
0,binary,Hybrid (quantum circuit),872,96,210,1178,4.6
1,binary,Classical twin (dense 8x8 + tanh),872,72,210,1154,4.5
2,binary,Tiny MLP (93 params),872,93,210,1175,4.6
3,binary,Random Fourier map,872,90,210,1172,4.6
4,binary,Identity (no bottleneck),872,0,210,1082,4.2
5,multiclass,Hybrid (quantum circuit),3624,96,3048,6768,26.4
6,multiclass,Classical twin (dense 8x8 + tanh),3624,72,3048,6744,26.3
7,multiclass,Tiny MLP (93 params),3624,93,3048,6765,26.4
8,multiclass,Random Fourier map,3624,90,3048,6762,26.4
9,multiclass,Identity (no bottleneck),3624,0,3048,6672,26.1


In [3]:
# the encoder and decoder must be identical across variants (only the bottleneck changes)
for task in ["binary", "multiclass"]:
    counts = {v: build_model(task, v).param_breakdown() for v in ["full", "linear", "mlp96", "fourier", "identity"]}
    enc = {c["encoder"] for c in counts.values()}; dec = {c["decoder"] for c in counts.values()}
    print(task, "encoder sizes:", enc, " decoder sizes:", dec)
    assert len(enc) == 1 and len(dec) == 1

binary encoder sizes: {872}  decoder sizes: {210}
multiclass encoder sizes: {3624}  decoder sizes: {3048}
